## Let's build our own `csv` parser!

Here's a sample of a simple csv file:
```csv
name,age,height,employed
alice,35,1.64,true
bob,42,1.75,true
charlie,68,1.60,false
```
This is available at `data/people.csv`.

You can try this yourself. If you're new to Python, you probably need to read a few docs to know which functions to use. Consider how the `csv` data would be represented in Python. (with `list`s? with `dict`s?)

Before we start, we need to consider the problem of knowing if a `csv` value is a string, float, integer or boolean (we will only consider these datatypes for this toy problem.

This is a scuffed solution, but we can try one case and proceed to the next if Python raises an error (using `try`/`except` blocks).

In [ ]:
def dyn_convert(s):
    if s == 'true': return True
    elif s == 'false': return False
    
    try: return int(s)
    except: pass

    try: return float(s)
    except: pass
    
    return s

Then, we read the file and parse it. The first line we use as the keys, and the subsequent lines are the values. To deal with multiple columns, we define a list of lists that will contain the values, and assign each of the lists to its respective key.

In [ ]:
def read_csv(filepath):
    with open(filepath, 'r') as f:
        csvlines = f.readlines()

    keys = csvlines[0].strip().split(',')

    # Create n DIFFERENT lists for n columns
    # If you use [[]] * n, all the lists are actually the same (references the same list)
    columns = [[] for _ in range(len(keys))]
    
    for line in csvlines[1:]:
        row_vals = [dyn_convert(s) for s in line.strip().split(',')]
        
        for col, row_val in zip(columns, row_vals):
            col.append(row_val)

    return {key: col for key, col in zip(keys, columns)}


In [ ]:
read_csv('data/people.csv')

Then we can write a function that, for example, takes the parsed `csv` output and a condition to query an object:

In [ ]:
def query(data, cond):
    data_count = len(data[list(data)[0]])

    results = []
    
    for i in range(data_count):
        def obj(k):
            return data[k][i]
        if cond(obj):
            results.append({k: obj(k) for k in data})

    return results

In [ ]:
people = read_csv('data/people.csv')

query(people, lambda obj: not obj('employed'))

# Pandas

This part of the lesson is based on [the official 10 minutes to pandas user guide](https://pandas.pydata.org/docs/user_guide/10min.html#)

To start, make sure `pandas` is installed and import it:

In [ ]:
import pandas as pd # To import pandas

# Other libraries needed
import numpy as np

`pandas` is a library that is used to process tables of data.

There are 3 main data structures to understand in `pandas`:

* **Scalar**: A primitive value, such as integers, floating point values, strings, etc.
* **Series**: A 1D array of scalar values
* **Dataframe**: A group of series represented using a 2D table

Each element of a series or row of a dataframe can be labelled with an index, otherwise, the index defaults to the integer position of the element. We shall see why this is important when we create dataframes out of multiple series.

## Series

In [ ]:
# Basic series
pd.Series([1, 3, 4, 2])

In [ ]:
# Series with index
s1 = pd.Series([5.5, 1.2, 7.8, -0.4], index=['a', 'b', 'c', 'd'])
print(s1)
print()

# Access element with an index
print('s1[\'b\'] =', s1['b'])

# Set element in series
print('before setting, s1[\'c\'] =', s1['c'])
s1['c'] = 6.8
print('after setting, s1[\'c\'] =', s1['c'])

You might or might not have noticed `pandas` uses `numpy` arrays under the hood. As such, `pandas` series can act as `numpy` arrays and undergo most of the same operations.

In [ ]:
s1 = pd.Series([5.5, 1.2, 7.8, -0.4])
s2 = pd.Series([-9.0, 4.5, 0.1, 1.2])

print('s1 + s2 =')
print(s1 + s2)
print()

print('s1 * s2 =')
print(s1 * s2)
print()

print('np.exp(1j * s1) =')
print(np.exp(1j * s1))

Here, the index is significant as elements in the series will only be operated upon each other if their indices match. If an index only appears on one of the operands, then the result will have that index as `NaN` to signify its nonexistence.

In [ ]:
s1 = pd.Series([5.5, 1.2, 7.8, -0.4], index=['a', 'b', 'c', 'd'])
s2 = pd.Series([-9.0, 4.5, 0.1, 1.2], index=['a', 'c', 'd', 'e'])

s1 + s2

## DataFrames

As stated before, dataframes are tables with series as their columns. Furthermore, each column will also have a name corresponding to the column header. This is important for organising data and querying data from the dataframe.

First let's create the above data into a `DataFrame`:

In [ ]:
data = [
    ('alice', 35, 1.64, True),
    ('bob', 42, 1.75, True),
    ('charlie', 68, 1.60, False),
]

df1 = pd.DataFrame(data, columns=['name', 'age', 'height', 'employed'])
df1

In [ ]:
# Alternatively, using Series and dicts:
df1 = pd.DataFrame({
    'name': pd.Series(['alice', 'bob', 'charlie']),
    'age': pd.Series([35, 42, 68]),
    'height': pd.Series([1.64, 1.75, 1.60]),
    'employed': pd.Series([True, True, False]),
})

df1

There are many ways to create `DataFrame`s, refer to the pandas docs for more examples.

### Importing data

One of the most appealing features of `pandas` is the ability to easily import and export data to files. For example:

In [ ]:
df1 = pd.read_csv('data/people.csv')

df1['name']

There are many formats that `pandas` can read from but they should broadly have the same output for the same dataset.

For the next sections, we shall use the slightly bigger `people35.csv` data.

In [ ]:
# The data has dates as indices so we set it as the index...
df35 = pd.read_csv('data/people35.csv', index_col=0)

# ... and convert it to actual dates
df35.index = pd.to_datetime(df35.index)

## Selecting and Filtering Data

First, inspect the data. We can see that it has dates as the index instead of integer or string ids. Take note of this.

We can select certain columns in the dataset by: (Selecting one column only will return the underlying Series)

In [ ]:
df35['weight']

In [ ]:
df35[['employed', 'height']] # Select multiple columns

To select certain rows by label (in this case, a date), we use the `loc` accessor.

In [ ]:
df35.loc['23/4/26 4:24:07']

We can pass a second argument to `loc` to obtain a certain column as well:

In [ ]:
df35.loc['23/4/26 4:24:07', 'weight']

`loc` also accepts ranges of values.

In [ ]:
df35.loc['23/4/26':'24/4/26']

`iloc` works similar to `loc` but uses integer indices of position regardless of the label. (Note that `iloc` also specifies columns with integer indices so if you still want to use the names, select them separately instead of using the `iloc` accessor.)

In [ ]:
df35.iloc[29]

In [ ]:
df35.iloc[5:10][['employed', 'height']] # Select certain columns for the 6th to 10th item of the data

In [ ]:
df35.iloc[::5] # Select every 5th item

In [ ]:
df35.iloc[[0, 12, 13, -1]] # Select the first, 12th, 13th, and last item

If you use a comparison operator on a `Series` (possibly selected from a `DataFrame`), it will return a `Series` where all the labels satisfying the condition are `True`, `False` otherwise.

In [ ]:
df35['weight'] > 90

Indexing a `DataFrame` with such a `Series` will filter out all the labels set to `False`, thus giving us a simple filtering mechanism

In [ ]:
df35[df35['weight'] > 90] # All data where 'weight' is greater than 90

In [ ]:
# Data where the height is greater than or equal to 1.75 and are not employed
# Note that we must use the element-wise logical operators:
# &, |, ~ instead of and, or, not
# Because we are dealing with pandas series, not boolean expressions
df35[ (df35['height'] >= 1.75) & ~df35['employed'] ]

### Altering Data

For this next section, we will operate on a copy of the original dataset. (so our original dataset doesn't get overwritten!)

In [ ]:
df35copy = df35.copy()

If you go back to above where we initially import the data:
```python
df35.index = pd.to_datetime(df35.index)
```
We do this because originally when we imported the data, the dates in the index are specified in strings instead of actual dates (`datetime64`). So to fix that, we convert the index to actual `datetime`s and reassign them back into the index.\

In general we can set entire columns as such:

In [ ]:
df35copy['height'] = 1.6
df35copy

In [ ]:
# Setting using an array
df35copy['weight'] = np.linspace(51, 85, 35)
df35copy

We can also set certain rows only with the same selection syntax like above:

In [ ]:
df35copy.loc['23/4/26 4:24:07', 'weight'] = -15.0
df35copy.loc['23/4/26 4:24:07']

In [ ]:
# Set a range of values:
df35copy.iloc[5:10, 0] = 100.0
df35copy.iloc[3:12]

In [ ]:
# Set using filtering:
df35copy.loc[~df35copy['employed'], 'weight'] = -68.0
df35copy

## Merging Data

A second round of data collection has been done, and the results were compiled in `people70.csv`.

There is a mistake though, the creator of the form forgot to set the fields as not optional. If you inspect the file, you'll notice that some of the cells have no data in them. Let's import the data anyway to see what happens.

In [ ]:
df70 = pd.read_csv('data/people70.csv', index_col=0)
df70.index = pd.to_datetime(df70.index)
df70

The cells that are missing data now have a 'not a number', `NaN` value in them. For good data analysis, we might need to remove them:

In [ ]:
df70 = df70.dropna(how='any')
df70

Now we have a cleaned dataset! We shall proceed to merging the two datasets. Notice how the second dataset has no `employed` column. So we only select the `height` and `weight` of the first dataset when merging.

In [ ]:
dfppl = pd.concat( (df35[['height', 'weight']], df70) )

## Data Analysis

`pandas` provides many convenience functions for data analysis. For example:

In [ ]:
print('Mean')
print(dfppl.mean(), '\n')

print('Standard Deviation')
print(dfppl.std(), '\n')

print('Correlation')
print(dfppl.corr(), '\n')

Here we can see, for example, that the height and weights are moderately correlated. Since the columns themselves are just `Series` that act like `numpy` arrays, we can do operations on them to calculate values such as the BMI:

In [ ]:
bmi_data = dfppl['weight'] / dfppl['height']**2
print('Mean:', bmi_data.mean())
print('stdev:', bmi_data.std())

[List of DataFrame statistics functions](https://pandas.pydata.org/docs/reference/frame.html#computations-descriptive-stats)

In this example, since the data is collected in two different rounds, we might want to see the statistics of each set individually. We notice that the first data is collected in April while the second in May. We can use the filtering methods to separate the data:

**NOTE: Be careful with writing dates because of how *ONE* country writes their dates.**

In [ ]:
dfppl_apr = dfppl[dfppl.index < '2026-05-01']
dfppl_may = dfppl[dfppl.index >= '2026-05-01']

In [ ]:
print('FIRST DATASET (APRIL)')
print(dfppl_apr.mean())
print(dfppl_apr.std())
print()

print('SECOND DATASET (MAY)')
print(dfppl_may.mean())
print(dfppl_may.std())

We can see that while the means are different, they are still within 1 S.D. of each other thus it is inconclusive whether the data collection is biased within themselves.

### Grouping

A better way to do the above separation is by grouping. To group, we create a `Series` (containing the labels) that have the categories we want to group by (e.g. the month), then we can use `groupby` to perform the grouping.

In [ ]:
grouped = dfppl.groupby(dfppl.index.month)
print(grouped.mean())
print(grouped.std())

Another thing we can group by is the height, e.g. into categories of 1.4x, 1.5x, 1.6x...

In [ ]:
dfppl_height_gb =dfppl['height'].groupby(dfppl['height'].round(decimals=1)) dfppl['height'].groupby(dfppl['height'].round(decimals=1))

Then we can get grouped data like the count:

In [ ]:
dfppl_height_gb.count()

## Plotting

`DataFrame`s have a builtin `plot` function. But for the sake of this lesson, we shall learn about the more manual method.

Since `Series` are `numpy` arrays under the hood, we can thus plot them directly using `matplotlib.pyplot`

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(dfppl['height'], dfppl['weight'], 'rx')
plt.grid()
plt.show()

We can even plot the height count data as a bar chart:

In [ ]:
height_counts = dfppl['height'].groupby(dfppl['height'].round(decimals=1)).count()

plt.bar(height_counts.index, height_counts, width=0.08)
plt.grid()
plt.show()

## Outputting Data

Once we process, filter the data, we will want to save it for later access. In this case, `pandas` is very convenient in providing convenience functions to do just this:

In [ ]:
dfppl.to_csv('data/people_full.csv')

Yes! It is, in fact, that easy. Of course, there are many ways to output the data. One of my favorites is `to_latex`, which prints out the `DataFrame` as a complete table. (or in this case, the first 8 values)

In [ ]:
dfppl.iloc[:8].to_latex()

Of course, if you use another table package like `tabularray` (like I do), then you can just copy the main data part without the formatting.

## Pandas Example: Simulation

In this example problem, we will perform an $N$-body physics simulation and output the data into a `csv` file.

In `nbody.py`, there is code to a general $N$-body simulation. In particular, let's import the `nbody_sim` function for the main simulation.

In [ ]:
from nbody import nbody_sim

The function takes three arguments: `initials`, `steps`, `dt`. `steps` is the amount of `steps` to take in the simulation, `dt` is the time in between steps in units of days. `initials` specify the initial condition with a list of tuples representing each body, where the tuple has the following format:
```
(mass, (position_x, position_y), (velocity_x, velocity_y))

-- UNITS --
mass: Earth mass
position_x, position_y: Earth radii
velocity_x, velocity_y: Earth radii/day
```
Here is the example initial condition we shall use:

In [ ]:
initials = [
    (1.07, (0., 0.), (3., -3.)),
    (0.4, (15.472, -9.5), (0., 28.98)),
    (0.12, (30.682, 0.), (10., 28.32)),
]

The function returns the tuple `(xhist, yhist)` where the values have the following format:
```
xhist : [[x_00, x_01,... x_0N], [x_10, x_11,... x_1N],... [x_M0, x_M1,... x_MN]]
yhist : [[y_00, y_01,... y_0N], [y_10, y_11,... y_1N],... [y_M0, y_M1,... y_MN]]
```
where `(x_mn, y_mn)` are the `x` and `y` positions of body `m` at step `n` in units of Earth radii.

We can thus plot the positions as such:

In [ ]:
xhist, yhist = nbody_sim(initials, 500, 0.02)

colours = ['r', 'g', 'b']
for xs, ys, colour in zip(xhist, yhist, colours):
    plt.plot(xs, ys, '-', color=colour)

plt.gca().set_aspect('equal')
plt.grid()
plt.show()

Now that we've obtained the actual data, we can proceed with the actual problem. First we need to go through a few considerations on how to save the data.
 * First, we should leave the label as blank since there is nothing in the data that is guaranteed to be unique for each datapoint.
 * Second, the time for each datapoint should be recorded. But since it is not provided by `nbody_sim`, we need to calculate it ourselves based on the values of `steps` and `dt`.
 * Third, it might be tempting to save the data with columns of `x1`, `y1`, ... `xM`, `yM` where `M` is the number of bodies. But in general, we should make it so the number of columns remain constant for any given configuration. So instead, there should be a column `body` that denotes which body the datapoint belongs to (as an integer). And the `x` and `y` positions of each body is just aggregated into the `x` and `y` column.

With that in mind, we can now write the code to create the `DataFrame`:

In [ ]:
num_bodies = len(initials)
steps = 501 # including the first step
dt = 0.02

time_values = np.arange(steps) * dt

# First, create fragments of the final DataFrame for each body
df_fragments = []
for i, (xs, ys) in enumerate(zip(xhist, yhist)):
    frag = pd.DataFrame({'t': time_values, 'x': xs, 'y': ys})
    frag['body'] = i # Assign the `body` column all at once for each fragment
    
    df_fragments.append(frag)

# Finally, concatenate everything into the final DataFrame
# We ignore the index of the fragments to prevent collisions
df = pd.concat(df_fragments, ignore_index=True)

# Then we save the data
df.to_csv('data/nbody_sim.csv')

Then the next time we need the data again, we just read it back:

In [ ]:
df_nbody = pd.read_csv('data/nbody_sim.csv')

for body, df in df_nbody.groupby('body'): # Shorthand for df_nbody.groupby(df_nbody['body'])
    plt.plot(df['x'], df['y'], '-', label=f'body {body}')

plt.gca().set_aspect('equal')
plt.legend()How would you quantify this?
plt.grid()
plt.show()

## Pandas Exercise: Trains

`data/trains.csv` contains data collected at a train station on 19 May 2026. The train station has two platforms (1, 2) where trains go in opposite directions. The `time` column denotes the recorded time that a train arrives at the station. The `platform` column denotes which platform the train arrives on, and the `passengers` column denotes how many people are on that train at that time.

Use `pandas` to read in the data, then perform analysis on it. Comment on trends visible in the data. Some interesting points of consideration include:
* How many people rode on this line (past that station) that day?
* How many passengers were transported by hour of the day?
* Are there any surge of passenger count across the day? How would you quantify this?
* What did the operators of the line do to counteract the surge? How would you quantify this?
* Are there any differences in train ridership based on platform (and therefore, direction)? How would you quantify this?
* What might be the cause of the trends above?